# Python 2: collections

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-02-collections.ipynb)

Almost every pipeline moves records between collections: a list of rows, a dictionary keyed by customer, a set of IDs already seen. This chapter covers the built-in collections (`list`, `tuple`, `set`, `dict`), the helpers in the `collections` module (`defaultdict`, `Counter`, `namedtuple`, `deque`, `ChainMap`), and the mutability rules behind most subtle bugs.

Every section follows the same pattern. It states the rule, shows it in a code cell whose `assert` lines are the exact expected results, and then names where the rule shows up in data work. The notebook needs no installs. If an `assert` fails, your mental model of that rule is wrong, so read the comment next to it.

## Lists: ordered, indexed and mutable

A list is an ordered sequence that you can change. It allows duplicates, and every item has an **index**.

- Indexes start at `0`. `items[-1]` is the last item and `items[-2]` is the one before it.
- A **slice** `items[start:stop]` returns a **new** list that contains the items from `start` up to, but not including, `stop`. Leave `start` out to begin at the front, and leave `stop` out to run to the end.
- `append(x)` adds **one** item at the end, even when `x` is itself a list. `extend(iterable)` adds **each** item of an iterable at the end. `a + b` builds a new list and changes neither operand.
- `insert(i, x)` places `x` at index `i` and shifts later items right. `pop()` removes and returns the last item, and `pop(i)` removes the item at index `i`. Removing from the front is O(n) because every remaining item shifts left.

In [ ]:
rows = ["a", "b", "c", "d", "e"]
assert rows[0] == "a" and rows[-1] == "e" and rows[-2] == "d"

# A slice includes start and excludes stop. It is a copy, so changing it leaves rows alone.
assert rows[1:3] == ["b", "c"]
assert rows[:2] == ["a", "b"] and rows[3:] == ["d", "e"]
head = rows[:2]
head[0] = "z"
assert rows[0] == "a"

# append adds ONE item, even when that item is a list.
nested = [1, 2]
nested.append([3, 4])
assert nested == [1, 2, [3, 4]] and len(nested) == 3

# extend adds EACH item of the iterable.
flat = [1, 2]
flat.extend([3, 4])
assert flat == [1, 2, 3, 4]

# + builds a new list. Both operands are unchanged.
left, right = [1, 2], [3]
both = left + right
assert both == [1, 2, 3] and left == [1, 2] and right == [3]

# insert shifts later items right. pop removes and returns an item.
items = ["b", "c"]
items.insert(0, "a")
assert items == ["a", "b", "c"]
assert items.pop() == "c"        # no index: the last item
assert items.pop(0) == "a"       # index 0: the first item
assert items == ["b"]

### Iterating safely

A `for` loop visits items in order. Two rules prevent most loop bugs:

- **Never remove items from a list while looping over it.** Each removal shifts the rest left, but the loop index keeps moving forward, so the item that slid into the gap is skipped. Build a new list with a comprehension instead.
- **Use `enumerate` for positions and `zip` for parallel lists.** `enumerate(items)` yields `(index, item)` pairs, so you never keep a counter by hand. `zip(a, b)` pairs items by position, but it **stops at the shorter input and silently drops the rest**. `zip(a, b, strict=True)` (Python 3.10 and later) raises `ValueError` when the lengths differ.

### Where you'll use it

- **Batching records:** collect rows until a size limit, then write them in one call to a database or API.
- **Chunking large lists:** `batch[i:i + 500]` in a loop splits a long list into fixed-size chunks.
- **Ordered replay:** keep events in arrival order, because later steps often depend on that order.

In [ ]:
# Removing while iterating skips the item that slides into the removed slot.
evens = [2, 4, 6, 8]
for n in evens:
    if n % 2 == 0:
        evens.remove(n)
assert evens == [4, 8]           # 4 is even but survived: it moved into index 0 and was skipped

# Build a new list instead. The original is untouched.
nums = [1, 2, 3, 4]
odds = [n for n in nums if n % 2 == 1]
assert odds == [1, 3] and nums == [1, 2, 3, 4]

# enumerate gives the position without a manual counter.
labelled = [f"{i}:{name}" for i, name in enumerate(["ana", "ben"], start=1)]
assert labelled == ["1:ana", "2:ben"]

# zip stops at the shorter input, so id 3 is dropped without an error.
ids = [1, 2, 3]
names = ["ana", "ben"]
assert list(zip(ids, names)) == [(1, "ana"), (2, "ben")]

try:
    list(zip(ids, names, strict=True))
except ValueError:
    pass
else:
    raise AssertionError("strict=True should refuse inputs of different lengths")

## Sorting: `sorted`, `sort`, `key=` and stability

`sorted(items)` returns a **new** list and leaves `items` unchanged. `items.sort()` sorts **in place** and returns `None`, so `x = items.sort()` leaves `x` as `None`, which is a common bug. Both accept two options:

- `reverse=True` sorts in descending order.
- `key=` takes a function that returns the value to compare. Python sorts the original items by those values. For example, `key=lambda order: order["amount"]` means "compare orders by their amount". The items themselves are not changed.

**Stability** means that when two items have equal keys, they keep the order they had in the input. Sorting `[("ana", 2), ("ben", 1), ("cy", 2)]` by the number gives `ben`, `ana`, `cy`. Both `ana` and `cy` have the value 2, and `ana` came first in the input, so it still comes first. `reverse=True` keeps this rule too.

Stability is what makes multi-field sorts work. To sort by a primary field and then by a secondary field, sort by the **secondary** field first and the **primary** field last. The last sort keeps the secondary order inside each group of equal primary values.

In [ ]:
pairs = [("ana", 2), ("ben", 1), ("cy", 2)]
by_count = sorted(pairs, key=lambda pair: pair[1])
assert by_count == [("ben", 1), ("ana", 2), ("cy", 2)]        # ana stays before cy

assert sorted(pairs, key=lambda pair: pair[1], reverse=True) == [("ana", 2), ("cy", 2), ("ben", 1)]

# sorted() returns a new list. list.sort() changes the list and returns None.
numbers = [3, 1, 2]
assert sorted(numbers) == [1, 2, 3] and numbers == [3, 1, 2]
assert numbers.sort() is None and numbers == [1, 2, 3]

In [ ]:
orders = [
    {"id": 1, "amount": 50, "status": "paid"},
    {"id": 2, "amount": 20, "status": "open"},
    {"id": 3, "amount": 50, "status": "open"},
]

# Ids 1 and 3 both have amount 50. Id 1 came first in the input, so it stays first.
by_amount = sorted(orders, key=lambda o: o["amount"])
assert [o["id"] for o in by_amount] == [2, 1, 3]

# A tuple key compares the first element, and uses the second only to break ties.
by_amount_then_id = sorted(orders, key=lambda o: (o["amount"], o["id"]))
assert [o["id"] for o in by_amount_then_id] == [2, 1, 3]

# A numeric key can be negated to sort it descending inside a tuple key.
biggest_first = sorted(orders, key=lambda o: (-o["amount"], o["id"]))
assert [o["id"] for o in biggest_first] == [1, 3, 2]

In [ ]:
# Strings cannot be negated, so use two stable passes: secondary key first, primary key last.
rows = [("ana", "paid"), ("ben", "open"), ("cy", "paid"), ("dee", "open")]
rows.sort(key=lambda r: r[0])                     # secondary: name, A to Z
rows.sort(key=lambda r: r[1], reverse=True)       # primary: status, descending

assert rows == [("ana", "paid"), ("cy", "paid"), ("ben", "open"), ("dee", "open")]

# None cannot be compared with an int, so sorting a mix raises TypeError.
try:
    sorted([3, None, 1])
except TypeError:
    pass
else:
    raise AssertionError("None and int should not compare")

# Put None last by sorting on a key that returns a tuple: (is it missing?, value).
assert sorted([3, None, 1], key=lambda x: (x is None, x if x is not None else 0)) == [1, 3, None]

### Where you'll use it

- **Choosing the latest record:** sort by `(business_key, updated_at)` and keep the last row for each key.
- **Deterministic output:** sort before writing a file, so reruns produce identical files and tests compare reliably.
- **Top N:** `sorted(items, key=...)[:n]` works for small data. For a huge stream, `heapq.nlargest(n, items, key=...)` avoids sorting everything.

## Tuples: fixed-shape records

A tuple is an **immutable** sequence. You cannot add, remove or assign items. Use one for a record whose shape never changes, such as a `(key, value)` pair or a row from a query. These rules matter in practice:

- **Unpacking** assigns each element to its own name: `key, value = pair`. The number of names must equal the number of elements, or Python raises `ValueError`.
- **Star unpacking** `first, *middle, last = items` collects the rest into a list. The starred name is always a list, even when it is empty.
- **The comma makes the tuple,** not the parentheses. `("x")` is only the string `"x"`. `("x",)` is a one-item tuple.
- **Immutability is shallow.** A tuple cannot change which objects it holds, but a list stored inside it can still change.
- Swapping `a, b = b, a` works because Python builds the right-hand tuple before it assigns anything.

In [ ]:
key, value = ("customer_42", 180.5)
assert key == "customer_42" and value == 180.5

a, b = 1, 2
a, b = b, a
assert (a, b) == (2, 1)

first, *middle, last = [10, 20, 30, 40]
assert first == 10 and middle == [20, 30] and last == 40

head, *rest = ["only"]
assert head == "only" and rest == []

assert isinstance(("x",), tuple) and not isinstance(("x"), tuple)

try:
    x, y = (1, 2, 3)
except ValueError:
    pass
else:
    raise AssertionError("three values cannot unpack into two names")

point = (1, 2)
try:
    point[0] = 9
except TypeError:
    pass
else:
    raise AssertionError("tuples cannot be changed")

# The tuple's slots are fixed, but the list inside is still mutable.
bag = ([1], [2])
bag[0].append(99)
assert bag == ([1, 99], [2])

### `namedtuple`: a tuple with field names

`collections.namedtuple` builds a small tuple class whose fields have names. It keeps every tuple behaviour (immutable, unpackable, hashable), and reads like a record: `sale.amount` is clearer than `sale[2]`. Use it once a tuple has more than two or three fields.

### Where you'll use it

- **Database rows:** DB-API cursors return tuples. A named tuple lets you write `row.customer_id` instead of counting positions.
- **Composite keys:** `(customer_id, order_date)` works as a dictionary key or a set member when you need to deduplicate.

In [ ]:
from collections import namedtuple

Sale = namedtuple("Sale", ["store", "item", "amount"])
sale = Sale("oslo", "coffee", 4.5)

assert sale.amount == 4.5 and sale[0] == "oslo"     # attribute and index access both work
store, item, amount = sale                          # and it still unpacks
assert store == "oslo" and item == "coffee"
assert sale._asdict()["item"] == "coffee"

# _replace returns a changed copy. The original is not modified.
bigger = sale._replace(amount=5.0)
assert bigger.amount == 5.0 and sale.amount == 4.5

## Sets: unique items and fast membership

A set stores unique items with no defined order. Its main use is the membership test. `x in some_set` takes about the same time whatever the size of the set. `x in some_list` checks items one at a time, so it slows down as the list grows. Set items must be **hashable**, meaning immutable, such as numbers, strings and tuples of those. Lists and dicts cannot go in a set.

- `set()` makes an empty set. `{}` makes an empty **dict**.
- `add(x)` does nothing if `x` is already present. `discard(x)` removes `x` if present. `remove(x)` raises `KeyError` if it is missing.
- Set algebra: `a | b` is the union (in either), `a & b` the intersection (in both), `a - b` the difference (in `a` only), and `a ^ b` the symmetric difference (in exactly one). The method forms, such as `a.union(list)`, accept any iterable.
- `issubset`, `issuperset` and `isdisjoint` answer yes/no questions about two sets.
- A set has no order. Call `sorted(s)` before you print or write it, so the output is deterministic.
- `frozenset` is an immutable set. Because it is hashable, it can be a dictionary key or a member of another set.

In [ ]:
assert set() == set() and type({}) is dict       # {} is an empty dict, not a set

ids = {"e1"}
ids.add("e1")                                    # duplicate: no change
assert ids == {"e1"}
ids.discard("missing")                           # discard is silent when absent
try:
    ids.remove("missing")                        # remove raises when absent
except KeyError:
    pass
else:
    raise AssertionError("remove should raise KeyError for a missing item")

# Replayed messages: skip any ID that was already processed.
seen = set()
duplicates = []
for event_id in ["e1", "e2", "e1", "e3", "e2"]:
    if event_id in seen:
        duplicates.append(event_id)
    seen.add(event_id)
assert duplicates == ["e1", "e2"] and seen == {"e1", "e2", "e3"}

yesterday = {"a", "b", "c"}
today = {"b", "c", "d"}
assert yesterday | today == {"a", "b", "c", "d"}     # union
assert yesterday & today == {"b", "c"}              # intersection
assert yesterday - today == {"a"}                   # gone today
assert today - yesterday == {"d"}                   # new today
assert yesterday ^ today == {"a", "d"}              # in exactly one of the two days
assert {"b"}.issubset(yesterday) and yesterday.isdisjoint({"x"})

# frozenset is hashable, so it can be a dictionary key.
pair_key = frozenset({"b", "a"})
labels = {pair_key: "pair"}
assert labels[frozenset(["a", "b"])] == "pair"      # order inside the set does not matter

# The same question, two costs: membership in a list scans every item, in a set it does not.
big_list = list(range(100_000))
big_set = set(big_list)
assert 99_999 in big_list and 99_999 in big_set

### Where you'll use it

- **Deduplication:** keep the set of IDs already processed, so a replayed message is skipped.
- **Change detection:** the difference between yesterday's and today's keys gives the inserts and deletes for a snapshot comparison.
- **Access control:** checking whether a user's role set contains a required role.
- **Fast lookups in a loop:** convert a reference list to a set once before the loop, not inside it.

## Dictionaries: keyed lookup

A dictionary maps unique, hashable keys to values. Looking up a key is fast regardless of size. These are the rules to know:

- `d[key]` raises `KeyError` when the key is missing. `d.get(key, default)` returns `default` instead, and returns `None` when you give no default.
- `key in d` tests the **keys**, never the values.
- `d[key] = value` inserts a key or overwrites it. `del d[key]` removes a key. `d.pop(key, default)` removes the key and returns its value, or `default` when the key is missing.
- `d.keys()`, `d.values()` and `d.items()` are **live views**. They reflect later changes to `d` and are not copies.
- Dictionaries keep **insertion order** (guaranteed since Python 3.7). Iteration yields keys in the order they were first inserted.
- `d1 | d2` builds a merged dictionary. When both have a key, the value from `d2` wins. `d1.update(d2)` does the same merge in place.

In [ ]:
stock = {"apples": 4}
try:
    stock["pears"]
except KeyError:
    pass
else:
    raise AssertionError("a missing key should raise KeyError")

assert stock.get("pears") is None and stock.get("pears", 0) == 0
assert "apples" in stock and 4 not in stock          # `in` checks keys, not values

stock["pears"] = 3                                    # insert
stock["apples"] = 5                                   # overwrite
assert stock == {"apples": 5, "pears": 3}
assert stock.pop("pears") == 3 and "pears" not in stock
assert stock.pop("pears", None) is None               # the default avoids KeyError

# Views reflect later changes to the dictionary.
keys = stock.keys()
stock["plums"] = 1
assert "plums" in keys
assert list(stock) == ["apples", "plums"]             # insertion order

# Merging: the right-hand side wins on conflicting keys.
defaults = {"retries": 3, "timeout": 30}
overrides = {"timeout": 60}
merged = defaults | overrides
assert merged == {"retries": 3, "timeout": 60}
assert defaults == {"retries": 3, "timeout": 30}      # the originals are unchanged
defaults.update(overrides)
assert defaults["timeout"] == 60

In [ ]:
# Sort a dictionary by value: sort its items, then keep the order you need.
scores = {"ana": 7, "ben": 9, "cy": 7}
ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
assert ranked == [("ben", 9), ("ana", 7), ("cy", 7)]  # ana and cy tie; input order kept

# dict(ranked) turns the pairs back into a dictionary, which keeps this order.
assert list(dict(ranked)) == ["ben", "ana", "cy"]

### Where you'll use it

- **Enrichment in Python:** a dictionary built from a small reference table, used to look up a name for each event.
- **Column mapping:** a dictionary maps source field names to warehouse column names.
- **Config and settings:** most pipeline configuration is a dictionary loaded from YAML or JSON.

## Grouping and counting

Most pipeline work eventually asks, for each key, to collect or count something: events per user, orders per status, rows per day. In SQL that is `GROUP BY key` with `COUNT(*)`. In Python you use a dictionary in which each key maps to the group collected so far. The next four subsections build on each other: the plain `if` check comes first, then the helpers that remove it.

### Grouping by hand

Without a helper, you check whether the key is already present, create its list if it is not, and then append to that list. Read the `if` as "make sure the bucket exists, then put the item in it." The helpers below each perform this same check for you.

In [ ]:
events = [("ana", "click"), ("ben", "view"), ("ana", "view"), ("ana", "click")]

by_user = {}
for user, action in events:
    if user not in by_user:          # first time we see this user: create the bucket
        by_user[user] = []
    by_user[user].append(action)

assert by_user == {"ana": ["click", "view", "click"], "ben": ["view"]}

### `dict.setdefault(key, default)`

`setdefault` performs the check and the insert in one call. If the key exists, it returns the stored value and changes nothing. If the key is missing, it stores `default` under that key and returns it. So `d.setdefault(k, []).append(v)` is the one-line version of the `if` above.

**Gotcha:** the default object is built on every call, even when the key already exists. For an empty list that costs almost nothing. For an expensive object, it is wasted work on every call.

In [ ]:
by_user = {}
for user, action in events:
    by_user.setdefault(user, []).append(action)

assert by_user == {"ana": ["click", "view", "click"], "ben": ["view"]}

# It returns the stored value and leaves an existing value alone.
config = {"retries": 3}
assert config.setdefault("retries", 5) == 3      # key exists: 3 stays, 5 is ignored
assert config.setdefault("timeout", 30) == 30    # key missing: stored, then returned
assert config == {"retries": 3, "timeout": 30}

### `collections.defaultdict(factory)`

A `defaultdict` behaves like a dictionary, except that when you **read** a missing key, it calls `factory()`, stores the result under that key, and returns it. `defaultdict(list)` gives each new key an empty list, so the grouping loop has no `if` at all. The factory is called only for missing keys, so the wasted work that `setdefault` does on every call does not happen here.

Two things to know:

- **Reading a missing key creates it.** `d["zed"]` inserts `"zed"`. To look without inserting, use `in` or `.get()`.
- **The factory decides the default.** `defaultdict(int)` starts every key at `0`, which makes a counter. `defaultdict(set)` starts every key at an empty set.

In [ ]:
from collections import defaultdict

by_user = defaultdict(list)
for user, action in events:
    by_user[user].append(action)

assert by_user["ana"] == ["click", "view", "click"]

# `in` and .get() do not create a key. A read with [] does.
assert "zed" not in by_user
assert by_user.get("zed") is None
assert "zed" not in by_user
_ = by_user["zed"]                               # this read inserts an empty list
assert "zed" in by_user and by_user["zed"] == []

# defaultdict(int) as a counter: a missing key starts at 0.
clicks = defaultdict(int)
for user, action in events:
    if action == "click":
        clicks[user] += 1
assert clicks == {"ana": 2}                      # ben has no clicks, so no key was created

### `collections.Counter(iterable)`

A `Counter` is a dictionary built for counting. Give it any iterable and it returns `{item: how many times it appeared}`. It is the Python form of `GROUP BY item` with `COUNT(*)`. A missing item returns `0` instead of raising `KeyError`, and the lookup does not insert the key, so `counts["nobody"]` is always safe.

In [ ]:
from collections import Counter

actions = [action for _, action in events]
counts = Counter(actions)
assert counts == Counter({"click": 2, "view": 2})
assert counts["click"] == 2
assert counts["purchase"] == 0                   # a missing item counts as zero
assert "purchase" not in counts                  # and the lookup did not insert it

### `Counter.most_common(n)`

`most_common(n)` returns the `n` most frequent `(item, count)` pairs, highest count first. Called with no argument, it returns every item. Items with equal counts keep the order in which they were first seen.

In [ ]:
words = ["spark", "kafka", "spark", "dbt", "kafka", "spark"]
assert Counter(words).most_common(2) == [("spark", 3), ("kafka", 2)]
assert Counter(words).most_common() == [("spark", 3), ("kafka", 2), ("dbt", 1)]

# Ties keep first-seen order: "b" appears before "a" in the input.
assert Counter(["b", "a", "b", "a"]).most_common() == [("b", 2), ("a", 2)]

### `Counter` arithmetic: `update`, `+`, `-` and `subtract`

- `update(iterable)` adds the new counts into the existing ones, as `+=` would. It does not replace them.
- `c1 + c2` adds counts per item. Items with a total of zero or less are dropped.
- `c1 - c2` subtracts counts per item and **keeps only the positive results**. That makes it the way to ask "what grew?" when you compare two batches.
- `subtract(iterable_or_mapping)` subtracts in place and **keeps** negative counts.

In [ ]:
daily = Counter(["click", "view"])
daily.update(["click", "buy"])                   # adds to the existing counts
assert daily == Counter({"click": 2, "view": 1, "buy": 1})

yesterday = Counter({"click": 5, "view": 3, "buy": 1})
today = Counter({"click": 7, "view": 3})
growth = today - yesterday                       # zero and negative results are dropped
assert growth == Counter({"click": 2})

assert yesterday + today == Counter({"click": 12, "view": 6, "buy": 1})

adjusted = Counter({"click": 2})
adjusted.subtract({"click": 5})                  # in place, and negative counts are kept
assert adjusted["click"] == -3

### `Counter.elements()` and `Counter.total()`

`elements()` is the reverse of counting. It yields each item as many times as its count, which is useful for expanding a summary back into rows. `total()` adds up every count. It exists from Python 3.10 onward.

In [ ]:
summary = Counter({"click": 2, "view": 1})
assert sorted(summary.elements()) == ["click", "click", "view"]
assert summary.total() == 3

### Where you'll use it

- **Aggregation before the warehouse:** count events per user or per status when a full SQL engine is too heavy for a small file.
- **Data profiling:** `Counter` over a column shows its distinct values and their frequencies, the first step of a data-quality check.
- **Top-N reports:** `most_common(10)` gives the ten busiest products without sorting the whole table.

## Comprehensions

A comprehension builds a collection in one expression: `[expression for item in iterable if condition]`. Read it as "for each item, keep it if the condition holds, and produce the expression." The same shape builds sets (`{...}`), dictionaries (`{key: value ...}`) and lazy generators (`(...)`). Three rules:

- The `for` clauses run **left to right, outer to inner**, like nested loops. `for x in xs for y in ys` is the same as a loop over `xs` that contains a loop over `ys`.
- An `if` **at the end** filters: it keeps or drops an item. An `if ... else` **at the front** chooses a value: it always produces one item. The two are different and easy to confuse.
- A generator expression `(x for x in xs)` produces values one at a time. It does not build a list, so it suits large inputs when you only need to consume the values once.

Use a plain loop when the body needs several statements or the comprehension is hard to read.

In [ ]:
prices = {"apple": 0.5, "pear": 0.8, "fig": 2.0}

cheap = {item: price for item, price in prices.items() if price < 1.0}
assert cheap == {"apple": 0.5, "pear": 0.8}

names = ["ana", "ben", "ana"]
assert {name for name in names} == {"ana", "ben"}      # a set comprehension drops duplicates
assert [len(name) for name in names] == [3, 3, 3]

# Filter (if at the end) versus choose (if/else at the front).
values = [1, -2, 3]
assert [v for v in values if v > 0] == [1, 3]           # two items: -2 is dropped
assert [v if v > 0 else 0 for v in values] == [1, 0, 3] # three items: -2 becomes 0

# Nested for clauses: the first one is the outer loop.
pairs = [(x, y) for x in [1, 2] for y in ["a", "b"]]
assert pairs == [(1, "a"), (1, "b"), (2, "a"), (2, "b")]

# Flatten one level of nesting.
batches = [[1, 2], [3], []]
flat = [item for batch in batches for item in batch]
assert flat == [1, 2, 3]

# A generator expression: same syntax, with parentheses, evaluated lazily.
doubled_total = sum(v * 2 for v in values)
assert doubled_total == 4

### Nested data: reading a path safely

Real payloads, such as JSON from an API, are nested. `order["customer"]["address"]["city"]` raises `KeyError` as soon as any level is missing. Chaining `.get(key, {})` avoids that, but it has a gap: `.get` returns the default only for a **missing** key. If the key is present with the value `None`, the next `.get` fails. Use `or {}` to cover both cases.

In [ ]:
order = {"customer": {"address": {"city": "Oslo"}}}
city = order.get("customer", {}).get("address", {}).get("city", "unknown")
assert city == "Oslo"

# A missing level falls back to the default.
assert {}.get("customer", {}).get("address", {}).get("city", "unknown") == "unknown"

# A present key whose value is None breaks the plain chain, so use `or {}` at each level.
broken = {"customer": None}
try:
    broken.get("customer", {}).get("address")
except AttributeError:
    pass
else:
    raise AssertionError("None has no .get method")

safe = ((broken.get("customer") or {}).get("address") or {}).get("city", "unknown")
assert safe == "unknown"

### Where you'll use it

- **Flattening API payloads:** read nested JSON one level at a time with safe lookups, so one missing field does not fail the whole batch.
- **Building lookups:** `{row["id"]: row for row in rows}` indexes a table by its key for constant-time access.
- **Filtering a stream:** a comprehension keeps only the records that pass a rule, and it documents that rule in one line.

## Mutability, aliasing and copies

Assignment copies a **reference**, not the data. Two names can refer to the same list, so a change made through one name is visible through the other. This is called **aliasing**.

A **shallow copy** (`list(x)`, `x.copy()`, `dict(x)`, `x[:]`) creates a new outer container, but the objects inside are still shared. A nested list is shared between the original and the copy. `copy.deepcopy(x)` copies every level, so nothing is shared.

Default arguments follow the same logic. Python evaluates a default value once, when the function is defined, so a list default is shared across calls.

### Where you'll use it

- **Avoiding shared-state bugs:** a config dictionary that one job mutates will change the next job's settings unless it is copied first.
- **Snapshots:** keep a deep copy of a record before a transformation, so a test can diff the input against the output.

In [ ]:
import copy

a = [1, 2, 3]
b = a                     # the same list, under two names
b.append(4)
assert a == [1, 2, 3, 4]

c = list(a)               # a new outer list with the same items
c.append(5)
assert a == [1, 2, 3, 4]

grid = [[0, 0], [0, 0]]
shallow = list(grid)
shallow[0][0] = 9         # the inner lists are still shared
assert grid[0][0] == 9

deep = copy.deepcopy(grid)
deep[1][1] = 7
assert grid[1][1] == 0    # the deep copy does not touch the original

### Mutable default arguments

A default such as `bucket=[]` is created **once**, when the function is defined. Every call that relies on the default receives that same list, so items accumulate across calls. Use `None` as the default and create the object inside the function.

In [ ]:
def add_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

def add_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

assert add_bad("a") == ["a"]
assert add_bad("b") == ["a", "b"]     # the shared default still holds "a"
assert add_good("a") == ["a"]
assert add_good("b") == ["b"]         # a fresh list on every call

## `deque`: fast ends and a fixed-size window

`collections.deque` is a double-ended queue. Adding or removing at **either end** takes constant time. A `list` is fast at its end but slow at its front: `list.pop(0)` and `list.insert(0, x)` shift every item, which is O(n). Use a `deque` for a first-in, first-out queue. Set `maxlen` to keep only the most recent N items: when the deque is full, adding an item discards the oldest one from the other end.

In [ ]:
from collections import deque

queue = deque(["job1", "job2"])
queue.append("job3")                  # add on the right
assert queue.popleft() == "job1"      # take from the left: first in, first out
assert list(queue) == ["job2", "job3"]

# maxlen: a sliding window that keeps only the last three readings.
window = deque(maxlen=3)
for reading in [10, 20, 30, 40, 50]:
    window.append(reading)
assert list(window) == [30, 40, 50]
assert sum(window) / len(window) == 40

### `ChainMap`: layered configuration

`ChainMap(*maps)` searches several dictionaries in order and returns the first match. It does not copy them, so a change to a layer shows up immediately. Put the highest-priority layer first, for example command-line arguments, then environment variables, then defaults. Writes go only to the **first** dictionary.

### Where you'll use it

- **Configuration layers:** arguments override environment variables, which override defaults, with no merge step.
- **Deque windows:** a `deque(maxlen=n)` gives a rolling window over a stream of readings, such as the last hour of metrics.

In [ ]:
from collections import ChainMap

defaults = {"retries": 3, "timeout": 30}
environment = {"timeout": 60}
arguments = {}

config = ChainMap(arguments, environment, defaults)
assert config["timeout"] == 60 and config["retries"] == 3

arguments["retries"] = 5              # a change to a layer is seen at once
assert config["retries"] == 5

config["new"] = 1                     # writes go to the first map only
assert arguments == {"retries": 5, "new": 1}
assert "new" not in defaults

## Practice questions

1. Given a list of `(customer, amount)` pairs, return the total per customer as a dictionary.
2. Given two lists of IDs, return the IDs that appear in both, sorted and without duplicates.
3. Remove duplicates from a list while keeping the first occurrence of each item and the original order.
4. Given orders as dictionaries with `status` and `amount`, sort them by a custom status order (`open`, then `paid`, then `void`), and within each status put the largest amount first. Ties keep their input order.

## Answers

### Question 1

In [ ]:
def totals_by_customer(pairs):
    totals = {}
    for customer, amount in pairs:
        totals[customer] = totals.get(customer, 0) + amount
    return totals

assert totals_by_customer([("a", 5), ("b", 2), ("a", 3)]) == {"a": 8, "b": 2}

### Question 2

In [ ]:
def in_both(first, second):
    return sorted(set(first) & set(second))

assert in_both([3, 1, 1, 2], [2, 2, 3, 9]) == [2, 3]

### Question 3

In [ ]:
def dedupe_keep_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

assert dedupe_keep_order([3, 1, 3, 2, 1]) == [3, 1, 2]

### Question 4

In [ ]:
def sort_for_queue(orders):
    status_rank = {"open": 0, "paid": 1, "void": 2}
    return sorted(orders, key=lambda o: (status_rank[o["status"]], -o["amount"]))

orders = [
    {"id": 1, "status": "paid", "amount": 40},
    {"id": 2, "status": "open", "amount": 10},
    {"id": 3, "status": "open", "amount": 90},
    {"id": 4, "status": "void", "amount": 70},
    {"id": 5, "status": "paid", "amount": 40},
]
assert [o["id"] for o in sort_for_queue(orders)] == [3, 2, 1, 5, 4]

## Interview notes

- **`list.sort()` returns `None`.** It sorts in place. Use `sorted()` when you need a new list back.
- **Sorting is stable.** Equal keys keep their input order, including with `reverse=True`. A multi-field sort is a tuple key, or a sequence of stable sorts from the least significant field to the most significant.
- **Set and dict membership is O(1) on average.** `x in list` scans every item and is O(n). Converting a list to a set once turns a quadratic loop into a linear one.
- **Dicts keep insertion order** (Python 3.7 and later), and their `keys()`, `values()` and `items()` are live views.
- **`defaultdict` and `Counter` replace the `if key not in d` pattern.** `setdefault` builds its default on every call. `defaultdict` calls its factory only for missing keys. Reading a missing key with `[]` on a `defaultdict` inserts it, which can change a later `len()` or `in` result.
- **Shallow copies share nested objects.** Mutating a nested list through a shallow copy changes the original. Use `copy.deepcopy` when the data is nested.
- **Mutable default arguments are shared across calls.** Default to `None` and create the object inside the function.
- **Do not remove items from a list while looping over it.** Build a new list with a comprehension.
- **`zip` stops at the shorter input.** Use `strict=True` to fail loudly when the lengths differ.
- **`deque.popleft()` is O(1); `list.pop(0)` is O(n).** Use a `deque` for queues and sliding windows.